In [0]:
from pyspark.sql import functions as F

In [0]:
if spark.catalog.tableExists("`dataexpert-portfolio`.`market-data-tracker`.dim_market_activity"):
    existing_ids = spark.table("`dataexpert-portfolio`.`market-data-tracker`.dim_market_activity").select("id", "start_date", (F.concat(F.col("id"), F.lit("_"), F.col("start_date"))).alias("key"))
    existing_ids.createOrReplaceTempView("existing_ids")
    existing_ids.show()
else:
    pass

spark.read.table("`dataexpert-portfolio`.`market-data-tracker`.lb_watchlist_history").display()

In [0]:
market_watchlist_historical = spark.sql('''
          SELECT *,
          ROW_NUMBER() OVER (PARTITION BY symbol, email ORDER BY _timestamp DESC) AS activity_sort,
          LEAD(`_timestamp`) OVER (PARTITION BY symbol, email ORDER BY `_timestamp` ASC) AS end_date,
          concat(history._pg_xid, "_", history.updated_at) as key

          FROM `dataexpert-portfolio`.`market-data-tracker`.lb_watchlist_history history
          LEFT ANTI JOIN existing_ids exists ON
          concat(history._pg_xid, "_", history.updated_at) = exists.key
          ORDER BY symbol, email, _timestamp DESC
          ''')

market_watchlist_historical.show()

In [0]:
scd_new_schema = market_watchlist_historical.withColumn("tickerxuser_key", F.concat(F.col("symbol"), F.lit("_"), F.col("email")))

scd_new_schema = scd_new_schema.withColumn("is_active", F.when(F.col("activity_sort") == 1, True).otherwise(False))

scd_new_schema = scd_new_schema.select("symbol",F.col("latest_price").alias("price"), "email" ,F.col("_pg_change_type").alias("change_type"), F.col("_pg_xid").alias("id"),"_sort_by", "is_active","tickerxuser_key" ,F.col("updated_at").alias("start_date"), "end_date" )

scd_new_schema.show()

In [0]:
if scd_new_schema.count() == 0:
    print("No new records to insert")
    exit()
else:
    if spark.catalog.tableExists("`dataexpert-portfolio`.`market-data-tracker`.`dim_market_activity`"):
        scd_new_schema.write.mode("append").options(mergeSchema="true").saveAsTable("`dataexpert-portfolio`.`market-data-tracker`.`dim_market_activity`")
    else:
        scd_new_schema.write.mode("overwrite").saveAsTable("`dataexpert-portfolio`.`market-data-tracker`.`dim_market_activity`")